**Temperature**, **top_p**, **top_k**, and **max_tokens** all shape how the model picks its next token — but only one of them is about *length*; the other three are about *randomness*.
- **`temperature`** (0.0–1.0, default 1.0) — controls how much randomness gets injected into token selection; closer to **0.0** makes the model pick the highest-probability token almost every time, closer to **1.0** lets lower-probability tokens through for more variety
- **`top_p`** (nucleus sampling) — instead of a randomness *amount*, it sets a probability **cutoff**: only tokens whose combined probability adds up to `top_p` are even considered — a lower value narrows the pool to just the most likely options
- **`top_k`** — a cruder cutoff than `top_p`: only the **K** most probable tokens are considered at all, regardless of how confident the model actually is — Anthropic flags this as "advanced use only"; `temperature` alone covers most needs
- **Official rule: alter `temperature` OR `top_p`, never both** — current models can outright reject a request that sets both, so pick one lever and leave the other at default
- **`max_tokens`** is unrelated to randomness — it's a hard **ceiling** on response length, not a creativity dial, and it's the one parameter that's **required**, not optional
- For **consistent classification**, set **`temperature=0`** — it makes the model greedily pick the single most likely label every time instead of sampling, which is exactly what you want when the same email should always get the same answer
- Even at `temperature=0`, output isn't **fully guaranteed identical** across calls — tiny floating-point/hardware variations can still occasionally flip a borderline decision, so "deterministic" here means *overwhelmingly consistent*, not mathematically airtight
- `top_p` and `top_k` are largely irrelevant for this use case — they exist to preserve creative variety, and classification has no creative dimension worth keeping

**Temperature** and **`top_p`** both affect "how many options the model considers," but they do it in a way that **gets tangled together** if you turn on both at once.
- **Temperature** acts **first** — think of it like turning a dial that either makes the model's favorite choice look *even more* favorite (low temperature), or makes the other choices look *more competitive* (high temperature)
- **`top_p`** acts **second** — it just looks at whatever the scores are *after* temperature already changed them, and says "keep adding choices from the top until they add up to this percentage"
- Since `top_p` only sees the scores *after* temperature has already reshaped them, the same `top_p` number can end up meaning completely different things depending on what temperature did first
- **Example:** if temperature is very low, the model's top choice might already hog almost all the score — so even with `top_p` set to "allow a few options," there's basically only one real option left anyway, and `top_p` ends up doing nothing
- With a different temperature, that same `top_p` number could suddenly let in a dozen options instead — so you can't really predict what `top_p` is doing unless you also know exactly what temperature already did
- Leaving the one you're not using at its **default** avoids this tangled mess — the one setting you actually care about behaves the simple, predictable way it's supposed to

Think of it like a **two-step filter**: temperature decides how "spread out" the scores are, and `top_p` decides how many of the top scores to let through — but since `top_p` reacts to whatever temperature already did, the **same `top_p` number can behave in totally different ways**.
- Imagine classifying the **same FD email** twice, with **`top_p = 0.7`** both times — only the **temperature** changes
- **Low temperature** sharpens the scores a lot: `FD = 92%`, `Multiple Category = 6%`, `Non-FD = 2%`
- With `top_p = 0.7`, the model adds up scores from the top until it passes 70% — `FD` alone is already **92%**, way past 70% — so only `FD` makes it into the pool. The model has just **one real choice**: `FD`. Totally consistent.
- **Same email, same `top_p`, but now high temperature** instead — it spreads the scores out: `FD = 45%`, `Multiple Category = 35%`, `Non-FD = 20%`
- Adding from the top this time: `45% + 35% = 80%`, which passes 70% — so **both `FD` and `Multiple Category`** make it into the pool now. The model has **two real choices** to randomly pick between, for the exact same email.
- **Same `top_p = 0.7` both times — but the first case picked from 1 option, the second from 2.** The number you set never changed; what it actually *does* changed, because temperature quietly reshaped the scores underneath it.
- That's the tangle: you can't look at `top_p = 0.7` and know what it'll actually do — you'd also need to know exactly what temperature did first. Leaving one of them at default avoids that guesswork entirely.

Even at `temperature=0`, output isn't **fully guaranteed identical** across calls. Why? 

This is about **floating-point math inside the GPU**, not Claude "deciding" to be random — even greedy (`temperature=0`) decoding can occasionally flip a coin-toss-close decision purely because of how the hardware computes numbers.
- At `temperature=0`, the model is supposed to always pick the **single highest-probability** token — no randomness, just "pick the biggest number"
- The catch: GPUs process many requests **in parallel batches**, and the order in which numbers get added up can shift slightly depending on what else is running alongside your request at that exact moment
- This creates **tiny rounding differences** in the computed probabilities — normally far too small to matter, like a difference of 0.0001%
- **Example:** imagine a genuinely borderline email — it has an FD reference number *and* a loan complaint — so the model's internal scores come out almost tied: `FD: 50.001%` vs `Multiple Category: 49.999%`
- On **Run 1**, tiny rounding noise nudges it to `FD: 50.0008%` vs `Multiple Category: 49.9992%` → **`FD`** wins, that's the label you get
- On **Run 2**, same email, same `temperature=0`, but the noise lands the other way: `FD: 49.9997%` vs `Multiple Category: 50.0003%` → now **`Multiple Category`** wins instead
- Once that **first** token flips, everything after it changes too — each new token is generated based on everything before it, so one tiny coin-toss can swing the *entire* label, not just one character
- **The takeaway:** for almost every email — where one label is clearly more likely than the others — `temperature=0` gives you the exact same answer every time. This only shows up on the rare email that's a genuine toss-up between two labels.

In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic
import anthropic

load_dotenv()  # reads .env in the current working directory
api_key = os.getenv("anthropic_api_key")
client = anthropic.Anthropic(api_key=api_key)
MODEL_ID = "claude-haiku-4-5-20251001"

FACTUAL_PROMPT = "What is the capital of India? Answer in one short sentence."
CREATIVE_PROMPT = "Describe the Bajaj Finance company in one creative sentence."

In [2]:
def ask(prompt, max_tokens=60, temperature=None, top_p=None, top_k=None):
    
    """One API call. Only passes the params that are actually set,
    so we never accidentally send BOTH temperature and top_p together
    (Anthropic rejects that combination on current models)."""
    
    kwargs = {
        "model": MODEL_ID,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if temperature is not None:
        kwargs["temperature"] = temperature
    if top_p is not None:
        kwargs["top_p"] = top_p
    if top_k is not None:
        kwargs["top_k"] = top_k

    response = client.messages.create(**kwargs)
    return response.content[0].text.strip(), response.stop_reason

In [3]:
def run_n_times(label, prompt, params, n=3):
    print(f"\n--- {label} ---")
    print(f"params: {params}")
    for i in range(n):
        text, stop_reason = ask(prompt, **params)
        print(f"  Run {i + 1}: {text!r}  (stop_reason={stop_reason})")

In [4]:
print("=" * 70)
print("1. max_tokens — hard ceiling on response length")
print("=" * 70)
run_n_times("max_tokens=8  (too small -> gets cut off)",
            CREATIVE_PROMPT, {"max_tokens": 8, "temperature": 0}, n=1)
run_n_times("max_tokens=60 (enough room -> finishes naturally)",
            CREATIVE_PROMPT, {"max_tokens": 60, "temperature": 0}, n=1)
# Watch stop_reason: "max_tokens" in the first run means truncated;
# "end_turn" in the second means it finished on its own.

1. max_tokens — hard ceiling on response length

--- max_tokens=8  (too small -> gets cut off) ---
params: {'max_tokens': 8, 'temperature': 0}
  Run 1: '# Bajaj Finance: The Financial'  (stop_reason=max_tokens)

--- max_tokens=60 (enough room -> finishes naturally) ---
params: {'max_tokens': 60, 'temperature': 0}
  Run 1: "# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education, and prosperity across India's bustling heartland."  (stop_reason=end_turn)


In [5]:
print("\n" + "=" * 70)
print("2. temperature on a FACTUAL question")
print("=" * 70)
run_n_times("temperature=0 (deterministic-leaning)",
            FACTUAL_PROMPT, {"max_tokens": 30, "temperature": 0})
run_n_times("temperature=1 (max randomness)",
            FACTUAL_PROMPT, {"max_tokens": 30, "temperature": 1})


2. temperature on a FACTUAL question

--- temperature=0 (deterministic-leaning) ---
params: {'max_tokens': 30, 'temperature': 0}
  Run 1: 'The capital of India is New Delhi.'  (stop_reason=end_turn)
  Run 2: 'The capital of India is New Delhi.'  (stop_reason=end_turn)
  Run 3: 'The capital of India is New Delhi.'  (stop_reason=end_turn)

--- temperature=1 (max randomness) ---
params: {'max_tokens': 30, 'temperature': 1}
  Run 1: 'The capital of India is New Delhi.'  (stop_reason=end_turn)
  Run 2: 'The capital of India is New Delhi.'  (stop_reason=end_turn)
  Run 3: 'The capital of India is New Delhi.'  (stop_reason=end_turn)


In [6]:
print("\n" + "=" * 70)
print("3. temperature on a CREATIVE prompt")
print("=" * 70)
run_n_times("temperature=0 (deterministic-leaning)",
            CREATIVE_PROMPT, {"max_tokens": 40, "temperature": 0})
run_n_times("temperature=1 (max randomness)",
            CREATIVE_PROMPT, {"max_tokens": 40, "temperature": 1})


3. temperature on a CREATIVE prompt

--- temperature=0 (deterministic-leaning) ---
params: {'max_tokens': 40, 'temperature': 0}
  Run 1: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)
  Run 2: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)
  Run 3: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)

--- temperature=1 (max randomness) ---
params: {'max_tokens': 40, 'temperature': 1}
  Run 1: "# Bajaj Finance: India's Financial Alchemist\n\nBajaj Finance 

In [7]:
print("\n" + "=" * 70)
print("4. top_p on a CREATIVE prompt (temperature left at default)")
print("=" * 70)
run_n_times("top_p=0.1  (narrow pool -> very few real choices)",
            CREATIVE_PROMPT, {"max_tokens": 40, "top_p": 0.1})
run_n_times("top_p=0.99 (wide pool -> almost everything allowed)",
            CREATIVE_PROMPT, {"max_tokens": 40, "top_p": 0.99})


4. top_p on a CREATIVE prompt (temperature left at default)

--- top_p=0.1  (narrow pool -> very few real choices) ---
params: {'max_tokens': 40, 'top_p': 0.1}
  Run 1: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)
  Run 2: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)
  Run 3: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)

--- top_p=0.99 (wide pool -> almost everything allowed) ---
params: {'max_tokens': 40, 'top_p': 0.99}
  Run 1: "# Bajaj Fina

In [8]:
print("\n" + "=" * 70)
print("5. top_k on a CREATIVE prompt (temperature left at default)")
print("=" * 70)
run_n_times("top_k=1   (only the single best token -> acts like temperature=0)",
            CREATIVE_PROMPT, {"max_tokens": 40, "top_k": 1})
run_n_times("top_k=100 (lots of tokens allowed in)",
            CREATIVE_PROMPT, {"max_tokens": 40, "top_k": 100})


5. top_k on a CREATIVE prompt (temperature left at default)

--- top_k=1   (only the single best token -> acts like temperature=0) ---
params: {'max_tokens': 40, 'top_k': 1}
  Run 1: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)
  Run 2: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)
  Run 3: '# Bajaj Finance: The Financial Alchemist\n\nBajaj Finance transforms everyday aspirations into accessible reality by turning credit into a golden thread that weaves together dreams of homes, education'  (stop_reason=max_tokens)

--- top_k=100 (lots of tokens allowed in) ---
params: {'max_tokens': 40, 'top_k': 100}
  Run 1: "# Bajaj Finan